# Detecção de Objetos

Um classificador responde **o que** há na imagem. Um detector responde **o que** há e **onde**: a saída é uma lista de **caixas delimitadoras** (*bounding boxes*), cada uma com uma classe e uma confiança, e a imagem pode ter vários objetos.

Vamos treinar do zero um detector inspirado no **YOLO** (*You Only Look Once*), que prevê todas as caixas de uma vez com uma rede convolucional. Os dados são sintéticos: dígitos do MNIST espalhados em uma imagem maior.

In [ ]:
import random

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## O dataset sintético

Cada imagem tem $112 \times 112$ pixels e de 1 a 6 dígitos do MNIST, com tamanhos entre 24 e 48 pixels, colados em posições sorteadas sem se sobrepor. Como somos nós que colamos os dígitos, as anotações vêm de graça.

Cada caixa é guardada como `(classe, x, y, w, h)`: $(x, y)$ é o centro e $(w, h)$ são a largura e a altura, todos divididos pelo tamanho da imagem, entre 0 e 1. É o formato usado pelo YOLO.

In [ ]:
IMAGE_SIZE = 112

mnist_train = datasets.MNIST(root="data", train=True, download=True, transform=transforms.ToTensor())
mnist_test = datasets.MNIST(root="data", train=False, download=True, transform=transforms.ToTensor())


def overlaps(a, b):
    ax, ay, a_size = a
    bx, by, b_size = b
    return ax < bx + b_size and bx < ax + a_size and ay < by + b_size and by < ay + a_size


def make_image(mnist, rng, max_digits=6):
    image = torch.zeros(1, IMAGE_SIZE, IMAGE_SIZE)
    placed = []   # (x0, y0, tamanho) em pixels
    boxes = []    # (classe, x, y, w, h) entre 0 e 1

    n_digits = rng.randint(1, max_digits)
    for _ in range(100):   # tentativas
        if len(boxes) == n_digits:
            break

        size = rng.randint(24, 48)
        x0 = rng.randint(0, IMAGE_SIZE - size)
        y0 = rng.randint(0, IMAGE_SIZE - size)
        if any(overlaps((x0, y0, size), other) for other in placed):
            continue

        digit, label = mnist[rng.randrange(len(mnist))]                             # [1, 28, 28]
        digit = F.interpolate(digit[None], size=(size, size), mode="bilinear")[0]   # [1, size, size]
        image[:, y0:y0 + size, x0:x0 + size] = digit

        placed.append((x0, y0, size))
        x = (x0 + size / 2) / IMAGE_SIZE
        y = (y0 + size / 2) / IMAGE_SIZE
        boxes.append((label, x, y, size / IMAGE_SIZE, size / IMAGE_SIZE))

    return image, boxes


def draw_boxes(ax, boxes, color, show_labels=True):
    for box in boxes:
        label, x, y, w, h = box[0], *box[-4:]   # aceita também (classe, confiança, x, y, w, h)
        x0 = (x - w / 2) * IMAGE_SIZE
        y0 = (y - h / 2) * IMAGE_SIZE
        ax.add_patch(patches.Rectangle((x0, y0), w * IMAGE_SIZE, h * IMAGE_SIZE,
                                       linewidth=2, edgecolor=color, facecolor="none"))
        if show_labels:
            ax.text(x0, y0, str(label), color="white", backgroundcolor=color, fontsize=8)


rng = random.Random(0)
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax in axes:
    image, boxes = make_image(mnist_train, rng)
    ax.imshow(image[0], cmap="gray")
    draw_boxes(ax, boxes, "green")
    ax.axis("off")
plt.show()

## A grade do YOLO

A rede tem uma saída de tamanho fixo, mas cada imagem tem um número diferente de objetos. O YOLO divide a imagem em uma **grade** de $S \times S$ células, e cada célula responde pelo objeto cujo **centro** cai dentro dela. Cada célula prevê:

- $p$: a confiança de que há um objeto centrado nela;
- $x, y$: o centro do objeto, **relativo à célula**, entre 0 e 1;
- $w, h$: a largura e a altura, relativas à imagem;
- $c_0, \dots, c_9$: os escores das 10 classes.

Com $S = 7$, cada célula cobre $16 \times 16$ pixels e o alvo é um tensor $7 \times 7 \times 15$. Um objeto com centro $(x, y)$ cai na célula da linha $i$ e coluna $j$:

$$
i = \lfloor y \cdot S \rfloor, \qquad j = \lfloor x \cdot S \rfloor, \qquad x_{cel} = x \cdot S - j, \qquad y_{cel} = y \cdot S - i
$$

Cada célula guarda um objeto só. Aqui isso basta: como os dígitos não se sobrepõem e têm pelo menos 24 pixels, mais que o lado de uma célula, dois centros não caem na mesma célula.

A função `grid_to_boxes` faz o caminho inverso, mantendo as células com $p$ acima de um limiar. É ela que vamos usar na saída da rede.

In [ ]:
S = 7
NUM_CLASSES = 10


def boxes_to_grid(boxes):
    grid = torch.zeros(S, S, 5 + NUM_CLASSES)   # [7, 7, 15]
    for label, x, y, w, h in boxes:
        i = int(y * S)
        j = int(x * S)
        grid[i, j, 0] = 1                                    # p
        grid[i, j, 1:5] = torch.tensor([x * S - j, y * S - i, w, h])
        grid[i, j, 5 + label] = 1                            # classe em one-hot
    return grid


def grid_to_boxes(grid, threshold=0.5):
    boxes = []
    for i in range(S):
        for j in range(S):
            p = grid[i, j, 0].item()
            if p < threshold:
                continue
            x_cell, y_cell, w, h = grid[i, j, 1:5].tolist()
            label = grid[i, j, 5:].argmax().item()
            boxes.append((label, p, (j + x_cell) / S, (i + y_cell) / S, w, h))
    return boxes

O `Dataset` gera cada imagem na hora, sem nada salvo em disco. A semente do sorteio depende do índice, então a imagem `idx` é sempre a mesma. O treino usa dígitos do treino do MNIST e o teste, dígitos do teste.

In [ ]:
class SyntheticMNIST(Dataset):
    def __init__(self, mnist, num_images, seed):
        self.mnist = mnist
        self.num_images = num_images
        self.seed = seed

    def __len__(self):
        return self.num_images

    def __getitem__(self, idx):
        rng = random.Random(self.seed * 1_000_000 + idx)
        image, boxes = make_image(self.mnist, rng)
        return image, boxes_to_grid(boxes)


train_set = SyntheticMNIST(mnist_train, num_images=20000, seed=0)
test_set = SyntheticMNIST(mnist_test, num_images=1000, seed=1)

train_dataloader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2)
test_dataloader = DataLoader(test_set, batch_size=200)

images, targets = next(iter(train_dataloader))
print(images.shape)    # [64, 1, 112, 112]
print(targets.shape)   # [64, 7, 7, 15]

Desenhando as caixas recuperadas do alvo com `grid_to_boxes`, junto com as linhas da grade:

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(images[0, 0], cmap="gray")
for k in range(1, S):
    ax.axhline(k * IMAGE_SIZE / S - 0.5, color="gray", linewidth=0.5)
    ax.axvline(k * IMAGE_SIZE / S - 0.5, color="gray", linewidth=0.5)
draw_boxes(ax, grid_to_boxes(targets[0]), "green")
ax.axis("off")
plt.show()

## O detector

O detector não termina com `Flatten` e `Linear`: ele é **totalmente convolucional**. Quatro blocos de convolução e pooling reduzem a imagem de $112 \times 112$ para $7 \times 7$, o tamanho da grade, e cada posição do mapa final corresponde a uma célula.

A última camada é uma convolução $1 \times 1$, que aplica a mesma camada linear em cada posição e leva os 128 canais aos 15 valores da célula. O `permute` passa os canais para o fim, no formato do alvo. A rede devolve valores sem ativação, como os logits de um classificador.

In [ ]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )


class Detector(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(1, 16),     # [N, 16, 56, 56]
            conv_block(16, 32),    # [N, 32, 28, 28]
            conv_block(32, 64),    # [N, 64, 14, 14]
            conv_block(64, 128),   # [N, 128, 7, 7]
        )
        self.head = nn.Sequential(
            nn.Conv2d(128, 128, kernel_size=3, padding=1),   # [N, 128, 7, 7]
            nn.ReLU(),
            nn.Conv2d(128, 5 + NUM_CLASSES, kernel_size=1),  # [N, 15, 7, 7]
        )

    def forward(self, x):
        x = self.head(self.features(x))
        return x.permute(0, 2, 3, 1)                         # [N, 7, 7, 15]


model = Detector()
print(model(images).shape)   # [64, 7, 7, 15]

## A função de perda

A perda tem três partes, uma para cada pergunta que a célula responde:

- **tem objeto?** Entropia cruzada binária na confiança, em todas as células. Como a maioria das células está vazia, elas recebem o peso $\lambda_{noobj} = 0{,}5$, para a rede não aprender a dizer sempre "não";
- **onde está?** Erro quadrático nas coordenadas, passadas pela sigmoide, só nas células com objeto. Recebe o peso $\lambda_{coord} = 5$;
- **qual dígito?** Entropia cruzada nas classes, só nas células com objeto.

$$
\mathcal{L} = \lambda_{coord} \sum_{\text{obj}} \left\| b - \hat{b} \right\|^2
+ \sum_{\text{obj}} \text{BCE}(1, \hat{p})
+ \lambda_{noobj} \sum_{\text{noobj}} \text{BCE}(0, \hat{p})
+ \sum_{\text{obj}} \text{CE}(c, \hat{c})
$$

em que $b = (x, y, w, h)$ é a caixa, $\sum_{\text{obj}}$ soma sobre as células com objeto e $\sum_{\text{noobj}}$ sobre as vazias. O total é dividido pelo tamanho do lote.

In [ ]:
def detection_loss(predictions, targets, lambda_coord=5.0, lambda_noobj=0.5):
    has_object = targets[..., 0] == 1   # [N, 7, 7]
    pred = predictions[has_object]      # [n_obj, 15], só as células com objeto
    true = targets[has_object]

    confidence = F.binary_cross_entropy_with_logits(predictions[..., 0], targets[..., 0], reduction="none")
    object_loss = confidence[has_object].sum()
    no_object_loss = confidence[~has_object].sum()

    box_loss = F.mse_loss(pred[:, 1:5].sigmoid(), true[:, 1:5], reduction="sum")
    class_loss = F.cross_entropy(pred[:, 5:], true[:, 5:].argmax(dim=1), reduction="sum")

    total = lambda_coord * box_loss + object_loss + lambda_noobj * no_object_loss + class_loss
    return total / len(predictions)

## Treinamento

O laço é o mesmo dos classificadores, trocando a perda.

In [ ]:
def train(model, train_dataloader, num_epochs, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for images, targets in train_dataloader:
            images, targets = images.to(device), targets.to(device)
            loss = detection_loss(model(images), targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        losses.append(total_loss / len(train_dataloader))
        print(f"[{epoch + 1}/{num_epochs}] perda {losses[-1]:.3f}")

    return losses


num_epochs = 10

torch.manual_seed(42)
model = Detector().to(device)
losses = train(model, train_dataloader, num_epochs)

plt.plot(range(1, num_epochs + 1), losses)
plt.xlabel("época")
plt.ylabel("perda de treino")
plt.show()

## Das previsões às caixas

Na saída da rede, aplicamos a sigmoide na confiança e nas coordenadas e usamos a mesma `grid_to_boxes`. Só que um dígito grande ocupa várias células, e às vezes uma vizinha da célula responsável também acha que tem um objeto. O resultado são caixas repetidas sobre o mesmo dígito.

A **supressão de não máximos** (NMS, *non-maximum suppression*) remove as repetições: ordena as caixas pela confiança, guarda a primeira, apaga as da mesma classe que se sobrepõem muito a ela e repete com as que sobraram.

A sobreposição é medida pela **interseção sobre união** (IoU), a área em comum dividida pela área coberta pelas duas caixas. Vale 1 para caixas idênticas e 0 para caixas que não se tocam:

$$
\text{IoU}(A, B) = \frac{|A \cap B|}{|A| + |B| - |A \cap B|}
$$

In [ ]:
def iou(box_a, box_b):
    ax, ay, aw, ah = box_a
    bx, by, bw, bh = box_b

    # bordas do retângulo de interseção
    left = max(ax - aw / 2, bx - bw / 2)
    right = min(ax + aw / 2, bx + bw / 2)
    top = max(ay - ah / 2, by - bh / 2)
    bottom = min(ay + ah / 2, by + bh / 2)

    intersection = max(0, right - left) * max(0, bottom - top)
    union = aw * ah + bw * bh - intersection
    return intersection / union


def non_max_suppression(boxes, iou_threshold=0.5):
    boxes = sorted(boxes, key=lambda box: box[1], reverse=True)   # maior confiança primeiro
    kept = []
    while boxes:
        best = boxes.pop(0)
        kept.append(best)
        boxes = [box for box in boxes
                 if box[0] != best[0] or iou(box[2:], best[2:]) < iou_threshold]
    return kept


def detect(model, images, threshold=0.5, iou_threshold=0.5):
    model.eval()
    with torch.no_grad():
        grids = model(images.to(device)).cpu()     # [N, 7, 7, 15]
    grids[..., 0:5] = grids[..., 0:5].sigmoid()    # p, x, y, w, h entre 0 e 1
    return [non_max_suppression(grid_to_boxes(grid, threshold), iou_threshold) for grid in grids]

Contando as caixas no lote de teste com um limiar de confiança baixo, sem e com NMS. Com `iou_threshold=1`, a NMS não apaga nada. Depois, separamos as imagens em que a NMS apagou alguma caixa, ou seja, as que tinham caixas repetidas.

In [ ]:
images, targets = next(iter(test_dataloader))
n_true = sum(len(grid_to_boxes(target)) for target in targets)

without_nms = detect(model, images, threshold=0.1, iou_threshold=1.0)
with_nms = detect(model, images, threshold=0.1)
print(f"{n_true} dígitos")
print(f"sem NMS: {sum(map(len, without_nms))} caixas")
print(f"com NMS: {sum(map(len, with_nms))} caixas")

repeated = [k for k in range(len(images)) if len(with_nms[k]) < len(without_nms[k])]
print(f"{len(repeated)} imagens com caixas repetidas")

## Resultados

Em verde, as caixas verdadeiras; em vermelho, as detecções com o limiar baixo de 0,1, sem NMS na primeira linha e com NMS na segunda. As imagens são algumas das que tinham caixas repetidas: sem a NMS, uma célula vizinha da responsável também passa do limiar e aparece uma segunda caixa sobre o mesmo dígito.

Com o limiar padrão de 0,5, quase não sobra repetição para a NMS apagar. No treino, só a célula do centro tem $p = 1$ e as vizinhas são empurradas para $p = 0$ pelo termo $\lambda_{noobj}$, então a rede aprende a dar confiança alta a uma célula só por dígito. Nos YOLOs de verdade, com várias âncoras por célula, três escalas e cada objeto atribuído a mais de uma célula, as repetições são muito mais comuns e a NMS é indispensável.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))
for col, k in enumerate(repeated[:4]):
    for ax, boxes, name in [(axes[0, col], without_nms[k], "sem NMS"),
                            (axes[1, col], with_nms[k], "com NMS")]:
        ax.imshow(images[k, 0], cmap="gray")
        draw_boxes(ax, grid_to_boxes(targets[k]), "green", show_labels=False)
        draw_boxes(ax, boxes, "red")
        ax.set_title(f"{name}: {len(boxes)} caixas")
        ax.axis("off")
plt.tight_layout()
plt.show()

## Exercícios

### Exercício 1: limiares

Mude o `threshold` e o `iou_threshold` da `detect` e refaça a figura dos resultados. Com quais valores aparecem caixas repetidas sobre o mesmo dígito? Com quais o detector começa a perder dígitos?

### Exercício 2: uma grade mais fina

Troque a grade para $S = 14$. Quantos blocos de pooling a rede precisa ter? Ajuste o `Detector`, treine de novo e compare os resultados.

### Exercício 3: dígitos sobrepostos

Mude a `make_image` para permitir que os dígitos se sobreponham, usando `torch.maximum` para colar o dígito sem apagar o que já está na imagem. O que acontece quando dois centros caem na mesma célula? Quantas caixas a `boxes_to_grid` perde?